# 02 - Preprocessing

Section **3.3 Preprocessing** of the report.

Builds two model-ready datasets from the Wine Reviews v2 CSV (regression target: `points`):

- `data/processed/xgboost/` — unscaled tabular features for notebook 03
- `data/processed/nn/` — scaled tabular + TF-IDF text for notebook 04

No model training is performed here.

## Brief

- Drop redundant / structural columns (`region_2`, `taster_twitter_handle`, raw `title` after vintage extraction).
- Engineer numeric and binary features; target-encode high-cardinality fields (train-only CV).
- Stratified train / validation / test split on binned `points`.
- Export **two** processed datasets (XGBoost unscaled, NN scaled continuous + TF-IDF).
- Persist encoders, scaler, manifest under `data/processed/`.

In [ ]:
import json

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import StandardScaler

from diplo_mod_1.constants import INTERIM, PROCESSED, RANDOM_STATE, RAW
from diplo_mod_1.preprocessing import (
    DataCleaner,
    DataSplitter,
    DatasetExporter,
    TabularEncoder,
    TextEncoder,
)

## Step 1 — Cleaning

Drop `region_2` and `taster_twitter_handle`, extract `vintage_year` from `title`, impute `price` (median by `country` + `variety`) and key categoricals.

In [ ]:
df = DataCleaner.load(RAW / "winemag-data-130k-v2.csv")
print(f"Raw shape: {df.shape}")

cleaner = DataCleaner()
cleaner.fit(df)
cleaned = cleaner.clean(df)

INTERIM.mkdir(parents=True, exist_ok=True)
cleaned.to_parquet(INTERIM / "01_cleaned.parquet")
(INTERIM / "preprocessing_config.json").write_text(
    json.dumps(cleaner.config, indent=2),
    encoding="utf-8",
)
print("Missing after clean (top):")
print(cleaned.isnull().sum().sort_values(ascending=False).head(8))

## Step 2 — Base features

Non-target features: `log_price`, flags (`is_luxury`, `is_us`, `has_designation`, `price_missing`), `wine_age`, `description_length`.

In [ ]:
featured = cleaner.add_features(cleaned)
featured.to_parquet(INTERIM / "02_features.parquet")
featured[["log_price", "wine_age", "is_luxury", "description_length"]].describe()

## Step 3 — Train / validation / test split

Stratified on binned `points` (~64% / 16% / 20%). Indices are shared by both exported datasets.

In [ ]:
splitter = DataSplitter(random_state=RANDOM_STATE)
split_idx = splitter.split(featured)

PROCESSED.mkdir(parents=True, exist_ok=True)
np.savez(
    PROCESSED / "split_indices.npz",
    train=split_idx["train"],
    val=split_idx["val"],
    test=split_idx["test"],
)
for name, idx in split_idx.items():
    print(f"{name}: {len(idx):,} rows ({100 * len(idx) / len(featured):.1f}%)")

## Step 4 — Encode and export XGBoost dataset

Target encoding (`TargetEncoder`, `target_type='continuous'`, cv=5 on train), frequency maps, OHE for `taster_name` and `country`. Matrix is **unscaled**.

In [ ]:
train_df = featured.iloc[split_idx["train"]]
val_df = featured.iloc[split_idx["val"]]
test_df = featured.iloc[split_idx["test"]]
y_splits = {
    "train": train_df["points"].to_numpy(dtype=np.float32),
    "val": val_df["points"].to_numpy(dtype=np.float32),
    "test": test_df["points"].to_numpy(dtype=np.float32),
}

encoder = TabularEncoder()
x_xgb = {}
x_xgb["train"], feature_names_xgb, groups_xgb = encoder.fit_transform(train_df, y_splits["train"])
x_xgb["val"], _, _ = encoder.transform(val_df)
x_xgb["test"], _, _ = encoder.transform(test_df)

xgb_dir = PROCESSED / "xgboost"
DatasetExporter.export_xgboost(xgb_dir, x_xgb, y_splits, encoder, feature_names_xgb, groups_xgb)

meta = json.loads((xgb_dir / "feature_names.json").read_text(encoding="utf-8"))
print(f"XGBoost features: {len(meta['feature_names'])}")
print(f"X_train shape: {x_xgb['train'].shape}")
print("First 15 feature names:", meta["feature_names"][:15])

## Step 5 — Export NN dataset

Tabular branch: add `taster_strictness`, scale continuous columns with `StandardScaler` (fit on train). Text branch: TF-IDF on `description` (2000 features, fit on train).

In [ ]:
x_nn = {}
x_nn["train"], nn_names, nn_groups = encoder.transform(train_df)
x_nn["val"], _, _ = encoder.transform(val_df)
x_nn["test"], _, _ = encoder.transform(test_df)

scaler = StandardScaler()
scaler.fit(x_nn["train"][:, nn_groups["continuous"]])
x_nn_scaled = {}
for key, x in x_nn.items():
    arr = x.copy()
    arr[:, nn_groups["continuous"]] = scaler.transform(x[:, nn_groups["continuous"]])
    x_nn_scaled[key] = arr

text_encoder = TextEncoder()
text_encoder.fit(train_df)
x_txt = {key: text_encoder.transform(featured.iloc[idx]) for key, idx in split_idx.items()}

nn_dir = PROCESSED / "nn"
DatasetExporter.export_nn(
    nn_dir, x_nn_scaled, x_txt, y_splits, nn_names, nn_groups, scaler, text_encoder
)

print(f"NN tabular features: {len(nn_names)}")
print(f"X_tab_train shape: {x_nn_scaled['train'].shape}")
print(f"X_txt_train shape: {x_txt['train'].shape}, nnz={x_txt['train'].nnz:,}")

## Step 6 — Dataset manifest

Single JSON contract for notebooks 03–05.

In [ ]:
DatasetExporter.write_manifest(
    PROCESSED / "dataset_manifest.json",
    xgb_dir,
    nn_dir,
    PROCESSED / "split_indices.npz",
    {k: x_xgb[k].shape for k in x_xgb},
    {k: x_nn_scaled[k].shape for k in x_nn_scaled},
    {k: x_txt[k].nnz for k in x_txt},
)
manifest = json.loads((PROCESSED / "dataset_manifest.json").read_text(encoding="utf-8"))
display(manifest)

## Design notes (report)

**EDA-driven decisions (notebook 01):**

- `region_2` dropped (61% missing, structural); `is_us` retains geographic signal.
- `log_price` and `is_luxury` handle extreme price skew (Spearman ρ ≈ 0.61 with `points`).
- `taster_avg_points` (target-encoded) captures scorer bias (ANOVA η² ≈ 0.10).
- `vintage_year` dropped from feature matrix — perfectly collinear with `wine_age`; only `wine_age` is kept.
- `taster_name` and `country` removed from OHE — already represented as target-encoded floats, OHE would duplicate the signal.
- High-cardinality fields (`winery`, `variety`) use target / frequency encoding, not full one-hot.

**Leakage controls:** all encoders, frequency maps, TF-IDF, and scaler fit on **train** only.

**Two datasets:** XGBoost needs unscaled inputs; the NN needs normalized continuous columns and sparse text features. Training happens in notebooks 03 and 04 respectively.